# 04 — Ekstraksi Teks dengan OCR docTR

> Notebook ini adalah **hasil run asli di Kaggle** (GPU T4) lengkap dengan output-nya.

**Peran dalam workflow:** kedua model gambar masih sering tertukar pada kelas yang *tampilannya* mirip (form, invoice, scientific_report, presentation). Pembedanya ada pada *kata-kata* di halaman, jadi teks diekstrak dengan OCR berbasis neural network (docTR: detektor `db_resnet50` + pengenal `crnn_vgg16_bn`). Posisi tiap kata (bounding box) juga disimpan.

| | |
|---|---|
| Input | Data kompetisi |
| Output | `ocr_shard_*.jsonl.gz` (kata + kotak per halaman), `ocr_text.csv` |
| Durasi | ~5,6 jam (1,96 halaman/detik) |
| Hasil | 39.997 halaman (seluruh train + test) berhasil di-OCR |

Kode modul: `src/ocr.py`, `src/run_ocr.py`, `configs/ocr.yaml`.

# RVL-CDIP — GPU OCR with docTR

**Inputs:** the competition data. **Settings:** Accelerator **GPU T4**, Internet **ON** (pip install + model weights).

**How to run:**
1. `QUICK_RUN = True` → Run All. It OCRs 160 pages, prints the pages/second rate, the projected full-run time and sample text. **Read the sample text**: if it's mostly garbage, stop here, because text won't help.
2. `QUICK_RUN = False` → **Save Version → Save & Run All (Commit)**.

**Output (`/kaggle/working`):**
- `ocr_shard_*.jsonl.gz`: one record per page: `id`, `split`, `words`, `boxes` (x0,y0,x1,y1 scaled 0–1000, the LayoutLMv3 format), `conf`.
- `ocr_text.csv`: `id`, `split`, `n_words`, `mean_conf`, `text`. This is what the TF-IDF test uses.

**Resume:** the run stops itself before Kaggle's 12-hour limit. If it reports pages remaining, create a new version with **this notebook's own output attached as an input** and commit again. It skips pages already done.

## 1. Install and config

In [11]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-doctr"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'python-doctr'], returncode=0)

In [12]:
import os, gc, gzip, json, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
import doctr
from doctr.models import ocr_predictor

QUICK_RUN = False

DET_ARCH   = "db_resnet50"      # text detection
RECO_ARCH  = "crnn_vgg16_bn"    # text recognition
PAGES_PER_CALL = 16             # pages sent to the predictor at once
DET_BS, RECO_BS = 8, 512        # internal batch sizes for detection / recognition
SHARD_SIZE = 2000               # pages per output file
TIME_BUDGET_H = 11.0            # stop before Kaggle's 12h limit and leave time to write outputs
NUM_WORKERS = os.cpu_count() or 2
SEED = 42
OUT = Path("/kaggle/working")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("docTR", doctr.__version__, "| torch", torch.__version__)

Device: cuda Tesla T4
docTR v1.1.0 | torch 2.11.0+cu128


## 2. Locate data

In [13]:
cands = sorted(p for p in Path("/kaggle/input").rglob("train.csv"))
assert cands, "train.csv not found under /kaggle/input - did you add the competition data?"
ROOT = cands[0].parent
print("Data root:", ROOT)

train_df = pd.read_csv(ROOT / "train.csv")
test_df  = pd.read_csv(ROOT / "test.csv")

def resolve(rel):
    rel = str(rel)
    p1 = ROOT / rel
    if p1.exists():
        return str(p1)
    p2 = ROOT / Path(rel).parts[0] / rel      # e.g. ROOT/train/train/invoice/x.tif
    if p2.exists():
        return str(p2)
    return None

train_df["path"] = [resolve(f) for f in tqdm(train_df["filename"], desc="resolve train")]
test_df["path"]  = [resolve(f) for f in tqdm(test_df["filename"],  desc="resolve test")]
train_df["split"], test_df["split"] = "train", "test"
assert test_df["path"].notna().all(), "Some test files could not be located."

all_df = pd.concat([test_df[["id", "path", "split"]],
                    train_df.loc[train_df["path"].notna(), ["id", "path", "split", "class_name"]]],
                   ignore_index=True)

if QUICK_RUN:
    all_df = (train_df[train_df["path"].notna()].groupby("class_name").sample(n=10, random_state=SEED)
              [["id", "path", "split", "class_name"]].reset_index(drop=True))

print("Pages to consider:", len(all_df))

Data root: /kaggle/input/competitions/document-image-classification-sd234504


resolve train:   0%|          | 0/32003 [00:00<?, ?it/s]

resolve test:   0%|          | 0/7994 [00:00<?, ?it/s]

Pages to consider: 39997


## 3. Resume: skip pages already OCR'd
Shards from an earlier run of this notebook (attached as an input) are copied into the output, and their pages are skipped.

In [14]:
done = set()
if not QUICK_RUN:
    for f in sorted(Path("/kaggle/input").rglob("ocr_shard_*.jsonl.gz")):
        shutil.copy(f, OUT / f.name)
for f in sorted(OUT.glob("ocr_shard_*.jsonl.gz")):
    with gzip.open(f, "rt") as fh:
        done.update(json.loads(line)["id"] for line in fh)

todo = all_df[~all_df["id"].isin(done)].reset_index(drop=True)
print(f"Already done: {len(done)} | to do now: {len(todo)}")

Already done: 160 | to do now: 39837


## 4. OCR model
`assume_straight_pages=True` reads horizontal text only. Vertical text (mostly the Bates numbers along the margin) is skipped. That's deliberate: those numbers are noise for classification, and rotated-box mode is much slower.

In [15]:
try:
    predictor = ocr_predictor(det_arch=DET_ARCH, reco_arch=RECO_ARCH, pretrained=True,
                              assume_straight_pages=True, det_bs=DET_BS, reco_bs=RECO_BS)
except TypeError:
    print("det_bs/reco_bs not supported in this docTR version - using defaults")
    predictor = ocr_predictor(det_arch=DET_ARCH, reco_arch=RECO_ARCH, pretrained=True,
                              assume_straight_pages=True)
predictor = predictor.to(DEVICE).eval()

## 5. Run OCR
Pages are decoded on the CPU workers while the GPU runs OCR. Results are written in shards, so a crash or the time limit loses at most one shard.

In [16]:
class PageDataset(Dataset):
    def __init__(self, df):
        self.ids, self.paths = df["id"].tolist(), df["path"].tolist()
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        try:
            with Image.open(self.paths[i]) as im:
                im.seek(0)
                arr = np.asarray(im.convert("RGB"), dtype=np.uint8)
        except Exception:
            arr = np.full((1000, 760, 3), 255, dtype=np.uint8)   # unreadable page -> blank
        return self.ids[i], arr

loader = DataLoader(PageDataset(todo), batch_size=PAGES_PER_CALL, shuffle=False,
                    num_workers=NUM_WORKERS, collate_fn=lambda b: b)

split_of = dict(zip(todo["id"], todo["split"]))

def page_record(pid, page):
    words, boxes, conf = [], [], []
    for block in page.blocks:
        for line in block.lines:
            for w in line.words:
                (x0, y0), (x1, y1) = w.geometry
                words.append(w.value)
                boxes.append([int(np.clip(v, 0, 1) * 1000) for v in (x0, y0, x1, y1)])
                conf.append(round(float(w.confidence), 3))
    return {"id": pid, "split": split_of[pid], "words": words, "boxes": boxes, "conf": conf}

run_tag = time.strftime("%Y%m%d%H%M%S")
shard_idx, buffer = 0, []

def flush():
    global shard_idx, buffer
    if buffer:
        with gzip.open(OUT / f"ocr_shard_{run_tag}_{shard_idx:03d}.jsonl.gz", "wt") as fh:
            for r in buffer:
                fh.write(json.dumps(r) + "\n")
        shard_idx, buffer = shard_idx + 1, []

t0, n_done, stopped_early = time.time(), 0, False
with torch.inference_mode():
    for batch in tqdm(loader, desc="OCR"):
        ids, pages = zip(*batch)
        result = predictor(list(pages))
        buffer.extend(page_record(pid, pg) for pid, pg in zip(ids, result.pages))
        n_done += len(ids)
        if len(buffer) >= SHARD_SIZE:
            flush()
        if (time.time() - t0) / 3600 > TIME_BUDGET_H:
            stopped_early = True
            break
flush()

elapsed = time.time() - t0
rate = n_done / elapsed
print(f"OCR'd {n_done} pages in {elapsed/60:.1f} min -> {rate:.2f} pages/s")
if QUICK_RUN:
    print(f"Projected full run (~40k pages): {40000 / rate / 3600:.1f} h")
if stopped_early:
    print(f"Stopped at the time budget. {len(todo) - n_done} pages remain: "
          "attach this notebook's output as an input and commit again to resume.")

OCR:   0%|          | 0/2490 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:982: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


OCR'd 39837 pages in 337.9 min -> 1.96 pages/s


## 6. Consolidate and check OCR quality
Low word counts or low confidence in a class means OCR is failing there, and a text model won't help that class.

In [17]:
records = []
for f in sorted(OUT.glob("ocr_shard_*.jsonl.gz")):
    with gzip.open(f, "rt") as fh:
        records.extend(json.loads(line) for line in fh)

text_df = pd.DataFrame({
    "id":        [r["id"] for r in records],
    "split":     [r["split"] for r in records],
    "n_words":   [len(r["words"]) for r in records],
    "mean_conf": [float(np.mean(r["conf"])) if r["conf"] else 0.0 for r in records],
    "text":      [" ".join(r["words"]) for r in records],
}).drop_duplicates("id")
text_df.to_csv(OUT / "ocr_text.csv", index=False)
print("Pages in ocr_text.csv:", len(text_df), "| train", (text_df.split == "train").sum(),
      "| test", (text_df.split == "test").sum())

q = text_df.merge(train_df[["id", "class_name"]], on="id", how="inner")
summary = q.groupby("class_name").agg(
    pages=("id", "size"),
    median_words=("n_words", "median"),
    pct_under_5_words=("n_words", lambda s: round(100 * (s < 5).mean(), 1)),
    mean_conf=("mean_conf", "mean"),
).round(3).sort_values("median_words")
print(summary.to_string())

print("\n--- Sample OCR text (first 300 characters) ---")
for cls in ["form", "invoice", "scientific_report", "presentation", "questionnaire", "memo"]:
    sample = q[q["class_name"] == cls].head(1)
    if len(sample):
        print(f"\n[{cls}] {sample['text'].iloc[0][:300]}")

Pages in ocr_text.csv: 39997 | train 32003 | test 7994
                        pages  median_words  pct_under_5_words  mean_conf
class_name                                                               
file_folder              2022           2.0               67.8      0.552
presentation             1992          59.0               24.8      0.762
advertisement            2012          70.0                2.9      0.728
handwritten              2026          72.0                4.7      0.633
email                    2013          78.0                0.6      0.929
budget                   2004          94.0               14.6      0.788
invoice                  1982         115.5                4.6      0.810
scientific_report        1999         132.0                3.1      0.881
form                     2005         134.0                3.3      0.845
memo                     1994         186.0                0.9      0.921
letter                   1972         193.0              